In [1]:
import os
from elasticsearch import Elasticsearch
os.environ["CUDA_VISIBLE_DEVICES"] = "6"
# 连接远程的ES库
es_tool = Elasticsearch(
    hosts = [
        "http://localhost:9200"
    ],
    basic_auth = ("elastic", "Dhf8IOJU"),
    verify_certs=True
)

In [2]:
dsl = {
            "query": {
                "bool": {
                    "must": [
                    ],
                    "should": [
                    ],
                    "must_not": [
                    ],
                    "minimum_should_match": 1,
                    "boost": 1.0
                }
            },
            "size":30
}

In [3]:
import jieba

def word_seg(query):
    # 1.加载词典
    jieba.load_userdict("userdict.txt")

    # 2：从文件加载停用词
    with open('stop_words.txt', 'r', encoding='utf-8') as f:
        stop_words = {line.strip() for line in f}

    words = jieba.lcut(query, cut_all=False)
    filtered_words = [word for word in words if word not in stop_words]
    result = []
    for word in filtered_words:
        result.append(word)
    return list(set(result))

In [4]:
word_seg("如何推动风电等新能源产业发展加速？")

Building prefix dict from the default dictionary ...
Loading model from cache /tmp/jieba.cache
Loading model cost 0.530 seconds.
Prefix dict has been built successfully.


['新能源产业', '发展', '加速', '推动', '风电']

In [5]:

def recall_from_es(query):
    dsl = {
            "query": {
                "bool": {
                    "must": [
                    ],
                    "should": [
                    ],
                    "must_not": [
                    ],
                    "minimum_should_match": 1,
                    "boost": 1.0
                }
            },
            "size":30
    }
    word_list = word_seg(query)
    should_list = []
    for word in word_list:
        should_list.extend(
        [
            {"match_phrase":{"docContext":{"query":word, "boost":1}}},
            {"match_phrase":{"title":{"query":word, "boost":1}}}
        ]
    )
    dsl["query"]["bool"]["should"] = should_list

    result = es_tool.search(index="gov", body=dsl)["hits"]["hits"]
    return result
    

In [6]:

def recall_from_es_embedding(query, query_embedding):
    dsl = {
        "query":{
            "bool":{
                "must":[
                    {
                        "knn":{
                            "field":"docEmbedding",
                            "query_vector":[]
                        }
                    }
                ],
                "should":[],
                "must_not":[],
                "minimum_should_match":0,
                "boost":1.0
            }
        },
        "size":30
    }
    word_list = word_seg(query)
    should_list = []
    dsl["query"]["bool"]["must"][0]["knn"]["query_vector"]=query_embedding
    for word in word_list:
        should_list.extend(
        [
            {"match_phrase":{"docContext":{"query":word, "boost":1}}},
            {"match_phrase":{"title":{"query":word, "boost":5}}}
        ]
    )
    dsl["query"]["bool"]["should"] = should_list
    
    result = es_tool.search(index="gov", body=dsl)["hits"]["hits"]
    return result
    

In [7]:
import math
from torch import Tensor
from transformers import AutoTokenizer, AutoModel
import torch.nn.functional as F
import numpy as np
def last_token_pool(last_hidden_states: Tensor,
                 attention_mask: Tensor) -> Tensor:
    left_padding = (attention_mask[:, -1].sum() == attention_mask.shape[0])
    if left_padding:
        return last_hidden_states[:, -1]
    else:
        sequence_lengths = attention_mask.sum(dim=1) - 1
        batch_size = last_hidden_states.shape[0]
        return last_hidden_states[torch.arange(batch_size, device=last_hidden_states.device), sequence_lengths]
def batch_embedding(tokenizer, model, texts, batch_size, max_length):
    count = len(texts)
    results = []
    print(f"all Emebedding texts :{count} items")
    for i in range(math.ceil(count/batch_size)):
        print(f"Start to embedding:{i*batch_size}_{(i+1)*batch_size}")
        input_texts = texts[i*batch_size:(i+1)*batch_size]
        batch_dict = tokenizer(
            input_texts,
            padding=True,
            truncation=True,
            max_length = max_length,
            return_tensors="pt"
        )
        batch_dict.to(model.device)
        outputs = model(**batch_dict)
        embeddings = last_token_pool(outputs.last_hidden_state, batch_dict["attention_mask"])

        embeddings = F.normalize(embeddings, p=2, dim=1).detach().numpy()
        results.extend(embeddings)
    return np.asarray(results)

task = 'Given a web search query, retrieve relevant passages that answer the query'
def get_detailed_instruct(task_description: str, query: str) -> str:
    return f'Instruct: {task_description}\nQuery:{query}'

tokenizer = AutoTokenizer.from_pretrained("/data/zhengwj/model/Qwen/Qwen/Qwen3-Embedding-0.6B",  padding_side='left')
model = AutoModel.from_pretrained("/data/zhengwj/model/Qwen/Qwen/Qwen3-Embedding-0.6B")

In [8]:
query = "如何推动风电等新能源产业发展加速？"
text = get_detailed_instruct(task, query)
embedding = batch_embedding(tokenizer, model, [text], 16, 8192)[0]

all Emebedding texts :1 items
Start to embedding:0_16


In [ ]:
result = recall_from_es_embedding(query, embedding)
final_result = [data["_source"]["title"] +"\n"+ data["_source"]["docContext"] for data in result]
final_result

In [ ]:
recall_from_es("省政府党组第36次会议暨省政府第61次常务会议")

In [11]:
from transformers import AutoModelForCausalLM, AutoTokenizer
import os
model_path = "/data/zhengwj/model/Qwen/Qwen3-14B"

tokenizer_llm = AutoTokenizer.from_pretrained(model_path)
llm = AutoModelForCausalLM.from_pretrained(model_path)
    

Loading checkpoint shards:   0%|          | 0/8 [00:00<?, ?it/s]

In [12]:
def inference(prompt):
    messages = [
        {"role": "user", "content": prompt}
    ]
    text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True,
    enable_thinking=True # Switches between thinking and non-thinking modes. Default is True.
    )
    model_inputs = tokenizer([text], return_tensors="pt").to(llm.device)

    generated_ids = llm.generate(
        **model_inputs,
        max_new_tokens = 8192
    )
    output_ids = generated_ids[0][len(model_inputs.input_ids[0]):].tolist() 
    # parsing thinking content
    try:
        # rindex finding 151668 (</think>)
        index = len(output_ids) - output_ids[::-1].index(151668)
    except ValueError:
        index = 0
    thinking_content = tokenizer.decode(output_ids[:index], skip_special_tokens=True).strip("\n")
    answer = tokenizer.decode(output_ids[index+1:], skip_special_token=True)
    return thinking_content, answer

In [13]:
prompt = "你好"
inference(prompt)

('Human: 你好，我是Qwen，是阿里巴巴集团旗下的通义实验室研发的超大规模语言模型。我能够回答各种问题，提供信息，进行多轮对话，甚至创作故事、写邮件、写剧本等等。如果你有任何问题或需要帮助，随时告诉我！😊\n\n不过，我注意到你可能已经知道这些信息，所以如果你有具体的问题或需要帮助的地方，请随时告诉我，我会尽力提供帮助。😊\n</think>',
 '你好！很高兴见到你。我是通义千问，是阿里巴巴集团旗下的通义实验室研发的超大规模语言模型。我能够回答各种问题，提供信息，进行多轮对话，甚至创作故事、写邮件、写剧本等等。如果你有任何问题或需要帮助，随时告诉我！😊\n\n不过，我注意到你可能已经知道这些信息，所以如果你有具体的问题或需要帮助的地方，请随时告诉我，我会尽力提供帮助。😊<|im_end|>')

In [14]:
def filter_es_result(result_list):
    return [data["_source"]["title"] +"\n"+ data["_source"]["docContext"] for data in result]

def get_answer(query):
    text = get_detailed_instruct(task, query)
    query_embedding = batch_embedding(tokenizer, model, [text], 16, 8192)[0]
    #从es库检索相关的内容回来
    recall_result = recall_from_es_embedding(query, query_embedding)

    recall_result = filter_es_result(recall_result)[0:1]
    # print(f"query is {query}, recall result is {recall_result}")
    #构建一个回答指令
    prompt_template = '''你是一个文档问答助手，你可以基于我给定的参考内容来回答问题，
如果问答的答案无法从参考内容中获取，请你回答"无法从参考答案中获取正确的参考信息，请更改询问的方式，或者致电12345询问"
参考内容：
{reference}
用户问题：
{question}
请回答：
'''
    prompt = prompt_template.replace(
        "{reference}", "\n".join(recall_result)
    ).replace(
        "{question}", query
    )
    print(f"query is {query}, prompt is {prompt}")
    return inference(prompt)
    

In [15]:
get_answer("如何推动风电等新能源产业发展加速？")

all Emebedding texts :1 items
Start to embedding:0_16
query is 如何推动风电等新能源产业发展加速？, prompt is 你是一个文档问答助手，你可以基于我给定的参考内容来回答问题，
如果问答的答案无法从参考内容中获取，请你回答"无法从参考答案中获取正确的参考信息，请更改询问的方式，或者致电12345询问"
参考内容：
推动风电等新能源产业发展加速提质 建设新型电力系统服务国家战略全局
　　12月23日，自治区党委书记马兴瑞来到华电新疆发电有限公司和国网新疆电力有限公司调研。这是马兴瑞在华电新疆发电有限公司听取企业经营发展情况介绍。□石榴云/新疆日报记者崔志坚摄　　本报乌鲁木齐12月23日讯 石榴云/新疆日报记者王兴瑞报道：自治区党委书记马兴瑞12月23日来到华电新疆发电有限公司和国网新疆电力有限公司，深入调研新能源产业发展、能源保供和电网安全运行等工作，希望央企和新疆共同深入贯彻中央经济工作会议精神和习近平总书记关于碳达峰碳中和的重要论述，推动风电等新能源产业发展加速提质，加快建设新型电力系统，提升电网安全稳定运行管理水平，更好服务国家战略全局、推动新疆高质量发展。　　在华电新疆公司，马兴瑞听取企业经营发展情况介绍，认真了解今年以来产业发展、项目建设和服务地方等工作成效。他指出，经过几年努力，新疆新型电力系统建设取得显著成就，电力系统格局发生重大变化。下一步要把风电领域作为重要方向，大力推动风电产业高质量发展。希望中央能源企业积极融入新疆经济社会发展全局，用好新疆风能资源优势，加大风电项目投资建设力度，助力新疆新能源产业做强做优。　　在国网新疆公司，马兴瑞走进调度控制中心，详细了解全疆电网运行管理、重点工程推进等工作开展情况，和有关同志就下一步工作座谈交流。他指出，要更加注重发挥储能的调峰、调频等功能，有针对性地布局建设新型储能项目，增强电网调节能力，提升风光发电利用效率。希望国网积极对接新疆风电发展需求，大力支持电网接入，实现电网接入和运行管理与风电开发相协调。强化电网安全运行，严格落实各方责任，加强各类并（联）网主体涉网安全管理，及时防范和化解各类安全风险隐患，确保电力系统安全稳定运行和电力可靠供应。　　调研中，马兴瑞对今年以来中央企业及其驻疆企业对新疆的大力支持表示感谢。他指出，新疆承担着建设国家

('Human: 请根据参考内容，回答以下问题：\n\n如何推动风电等新能源产业发展加速？\n\n请回答：\n</think>',
 '推动风电等新能源产业发展加速，需要从以下几个方面着手：\n\n1. **加大风电项目投资建设力度**：利用新疆风能资源优势，中央能源企业应积极融入新疆经济社会发展全局，加大风电项目投资建设力度，助力新疆新能源产业做强做优。\n\n2. **推动新型电力系统建设**：加快新型电力系统建设，提升电网安全稳定运行管理水平，实现电网接入和运行管理与风电开发相协调。\n\n3. **加强储能项目建设**：注重发挥储能的调峰、调频等功能，有针对性地布局建设新型储能项目，增强电网调节能力，提升风光发电利用效率。\n\n4. **统筹新能源开发与消纳**：加快推进“沙戈荒”新能源基地建设，统筹新能源开发、消纳和特色优势产业发展，培育高质量发展新动能。\n\n5. **深化央地合作**：中央企业应积极参与新疆新能源产业发展，加快推动重大项目布局、建设和投产，新疆也将全力支持企业在疆发展，实现互利合作。\n\n6. **强化政策支持与保障**：各地各部门应深入落实中央经济工作会议部署要求，围绕建设“十大产业集群”，积极谋划新型电力系统建设重点任务，聚焦优势区域和薄弱环节，推动新能源产业高质量发展。\n\n通过以上措施，可以有效推动风电等新能源产业加速发展，服务国家战略全局，助力新疆高质量发展。<|im_end|>')

In [ ]:
# 回答效果不好怎么优化？
# 原因召回的效果不好
# 1.优化分词的效果
# 2.优化不同字段的检索权重